In [ ]:
#| default_exp suggest

# Suggest

> Deterministic keyword-overlap scorer that proposes 1-3 candidate Johnny Decimal categories for a file headed to `sorted/filesystem/` — no LLM, no network. Matches category name/tags/description from `config/d*.json` against the filename and preview text, and reports which words matched so you can judge the suggestion rather than trust it blindly.

In [ ]:
#| export
import re
from dataclasses import dataclass, field
from filesorter.domain import JDDomain, JDCategory

In [ ]:
#| export
STOPWORDS = {
    # Dutch
    "de", "het", "een", "en", "van", "voor", "met", "op", "in", "aan", "te", "dat", "die", "dit",
    "ook", "als", "maar", "wel", "niet", "wordt", "worden", "kan", "moet", "zijn", "ben", "bent",
    "hebben", "heeft", "deze", "dan", "dus", "om", "bij", "uit",
    # English — generic prose words that show up in descriptions but carry no category-specific signal
    "the", "and", "of", "for", "with", "from", "this", "that", "are", "was", "you", "your",
    "about", "above", "after", "again", "all", "also", "although", "any", "because", "been",
    "before", "being", "below", "between", "both", "but", "can", "could", "did", "does", "doing",
    "down", "during", "each", "few", "further", "had", "has", "have", "having", "here", "how",
    "into", "its", "itself", "just", "may", "might", "more", "most", "must", "not", "now", "off",
    "once", "only", "other", "our", "out", "over", "own", "shall", "she", "should", "some",
    "such", "than", "then", "there", "these", "they", "those", "through", "too", "under", "until",
    "very", "were", "what", "when", "where", "which", "while", "who", "whom", "why", "will",
    "would",
}

def _tokenize(text: str) -> set[str]:
    """Lowercase, split camelCase and non-alphanumeric runs into words, drop short/stop words."""
    spaced = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", text)
    words = re.findall(r"[a-zA-Z0-9]+", spaced.lower())
    return {w for w in words if len(w) >= 3 and w not in STOPWORDS}

In [ ]:
#| export
@dataclass
class Suggestion:
    """One candidate category for a file, with the words that justified it."""
    domain: JDDomain
    category: JDCategory
    score: int
    matched: list[str] = field(default_factory=list)

    @property
    def path(self) -> str:
        """`{domain_id}/{cat_id} {cat_name}`, same shape as `category_path()`."""
        return f"{self.domain.domain_id}/{self.category.id} {self.category.name}"

In [ ]:
#| export
def _match(tokens: set[str], tag_t: set[str], name_t: set[str], desc_t: set[str], weight: int) -> tuple[int, set[str]]:
    """Score `tokens` against one category's tag/name/description sets. Tag hits count 3x, name 2x, description 1x
    a plain word — curated tags are the strongest signal, incidental description text the weakest."""
    score, matched = 0, set()
    for t in tokens:
        if t in tag_t: score += weight * 3; matched.add(t)
        elif t in name_t: score += weight * 2; matched.add(t)
        elif t in desc_t: score += weight * 1; matched.add(t)
    return score, matched

In [ ]:
#| export
def suggest_categories(domains: list[JDDomain], filename: str, preview_text: str = "", top_n: int = 3) -> list[Suggestion]:
    """Rank categories across all `domains` by keyword overlap with `filename` (weighted 2x) and `preview_text` (1x) —
    a deliberately named file outweighs incidental content. Pass `path.stem` for `filename` to skip extension noise.
    Categories with zero overlap are dropped, so an empty result means "no signal, go browse manually" rather than
    a weak guess."""
    fname_tokens = _tokenize(filename)
    preview_tokens = _tokenize(preview_text) - fname_tokens
    results = []
    for domain in domains:
        for cat in domain.all_categories:
            tag_t, name_t, desc_t = _tokenize(" ".join(cat.tags)), _tokenize(cat.name), _tokenize(cat.description)
            s1, m1 = _match(fname_tokens, tag_t, name_t, desc_t, weight=2)
            s2, m2 = _match(preview_tokens, tag_t, name_t, desc_t, weight=1)
            score = s1 + s2
            if score > 0: results.append(Suggestion(domain, cat, score, sorted(m1 | m2)))
    results.sort(key=lambda s: -s.score)
    return results[:top_n]

## Tests

In [ ]:
from fastcore.test import *
from filesorter.domain import JDArea

def _fixture_domain() -> JDDomain:
    """A small synthetic domain: one category with curated tags, one with only a name."""
    return JDDomain(name="dX Test", areas=[JDArea(id="10-19", name="Test Area", categories=[
        JDCategory(id="11", name="Cloud Platforms", tags=["invoice", "facturen"], description="billing documents"),
        JDCategory(id="12", name="Invoice Archive", tags=[], description=""),
    ])])

# Tag match outranks name-only match, even though both match on the same word
domains = [_fixture_domain()]
results = suggest_categories(domains, filename="invoice_2026")
test_eq(len(results), 2)
test_eq(results[0].category.id, "11")   # tag hit: weight 2 * 3 = 6
test_eq(results[1].category.id, "12")   # name hit: weight 2 * 2 = 4
test_eq(results[0].matched, ["invoice"])

# Preview text carries a weaker signal (1x) than the filename (2x), but still surfaces the same winner
results2 = suggest_categories(domains, filename="report", preview_text="invoice facturen")
test_eq(results2[0].category.id, "11")
test_eq(set(results2[0].matched), {"invoice", "facturen"})

# No overlap anywhere → empty, not a weak guess
results3 = suggest_categories(domains, filename="xyz_qux_2026")
test_eq(results3, [])

print("✓ Weighting and empty-signal tests passed")

In [ ]:
# top_n caps the result count, still sorted highest score first
many = JDDomain(name="dY Test", areas=[JDArea(id="20-29", name="Area", categories=[
    JDCategory(id=str(i), name=f"Report Type {i}", tags=[], description="") for i in range(5)
])])
results4 = suggest_categories([many], filename="report_type", top_n=3)
test_eq(len(results4), 3)
scores = [r.score for r in results4]
test_eq(scores, sorted(scores, reverse=True))

# camelCase filenames split into separate words, same as snake_case or kebab-case
test_eq(_tokenize("CloudPlatformReport"), {"cloud", "platform", "report"})
test_eq(_tokenize("cloud-platform_report"), {"cloud", "platform", "report"})

print("✓ top_n and tokenizer tests passed")

In [ ]:
# Sanity check against the real config/ — most categories have no tags/description, only a name,
# so this is the realistic case: a filename that echoes a real category name should still surface it.
from filesorter.domain import load_domains
from pathlib import Path

def _repo_config_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / "config"
        if cand.is_dir() and any(cand.glob("d*.json")): return cand
    raise FileNotFoundError("config/ with d*.json not found above cwd")

real_domains = load_domains(_repo_config_dir())
real_results = suggest_categories(real_domains, filename="cloud-platforms-invoice")
test_eq(real_results[0].category.id, "34")
test_eq(real_results[0].domain.domain_id, "d6")
assert "cloud" in real_results[0].matched and "platforms" in real_results[0].matched

print(f"✓ Real-config sanity check passed — top pick: {real_results[0].path}")